# Домашнее задание: Multi-Branch MLP для Wine Quality

**Цель**: Реализовать multi-branch модель и добиться F1 score ≥ 40%

**Задачи**:
1. Реализовать три типа блоков: Bottleneck, Inverted Bottleneck, Regular
2. Создать Multi-Branch архитектуру
3. Использовать weighted loss для борьбы с дисбалансом классов
4. Подобрать оптимальные гиперпараметры (глубина, ширина, lr, оптимизатор)

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight
from wine_quality_data import WineQualityDataModule
from lightning_module import BaseLightningModule
from utils import set_seed
import pytorch_lightning as pl

sns.set_style('whitegrid')
set_seed(42)

## 1. Загрузка и анализ данных

Загрузим Wine Quality датасет и проанализируем распределение классов.

In [ ]:
# Загружаем данные
dm = WineQualityDataModule(batch_size=128)
dm.setup()

print(f'Train samples: {len(dm.train_dataset)}')
print(f'Val samples: {len(dm.val_dataset)}')
print(f'Input dim: {dm.input_dim}')
print(f'Num classes: {dm.n_classes}')

### 1.1. Анализ дисбаланса классов

Проанализируйте распределение классов и вычислите веса для weighted loss.

In [ ]:
# Получаем метки классов из train_dataset
train_labels = []
for i in range(len(dm.train_dataset)):
    _, label = dm.train_dataset[i]
    train_labels.append(label)
train_labels = np.array(train_labels)

# Строим гистограмму распределения классов
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(train_labels, bins=dm.n_classes, alpha=0.7, edgecolor='black', color='skyblue')
plt.xlabel('Class')
plt.ylabel('Frequency')
plt.title('Class Distribution in Training Set')
plt.xticks(range(dm.n_classes))
plt.grid(True, alpha=0.3)

# F1-оптимизированные веса
class_counts = np.bincount(train_labels)
n_classes = len(class_counts)

# УЛУЧШЕННЫЕ СТРАТЕГИИ ДЛЯ F1 MACRO
f1_strategies = {
    'f1_aggressive': (1.0 / np.power(class_counts, 0.3) / np.sum(1.0 / np.power(class_counts, 0.3)) * n_classes).astype(np.float32),
    'f1_boosted': (1.0 / np.power(class_counts, 0.4) / np.sum(1.0 / np.power(class_counts, 0.4)) * n_classes).astype(np.float32),
    'f1_extreme': (1.0 / np.power(class_counts + 1, 0.2) / np.sum(1.0 / np.power(class_counts + 1, 0.2)) * n_classes).astype(np.float32),
}

# Ручная настройка весов для максимизации F1
manual_weights = np.ones(n_classes, dtype=np.float32)

# Сильно усиливаем самые редкие классы
min_count_idx = np.argmin(class_counts)
second_min_idx = np.argsort(class_counts)[1]

manual_weights[min_count_idx] = 3.5
manual_weights[second_min_idx] = 2.5

# Умеренно усиливаем средние классы  
median_count = np.median(class_counts)
for i in range(n_classes):
    if class_counts[i] < median_count * 0.7 and class_counts[i] > np.min(class_counts):
        manual_weights[i] = 1.8

manual_weights = manual_weights / np.sum(manual_weights) * n_classes
f1_strategies['f1_manual'] = manual_weights.astype(np.float32)

# Визуализация весов
plt.subplot(1, 2, 2)
x = np.arange(n_classes)
width = 0.2

strategies_to_plot = ['f1_aggressive', 'f1_boosted', 'f1_manual']
colors = ['red', 'orange', 'purple']

for i, strategy in enumerate(strategies_to_plot):
    plt.bar(x + i*width, f1_strategies[strategy], width, label=strategy, alpha=0.8, color=colors[i])

plt.xlabel('Class')
plt.ylabel('Weight')
plt.title('Aggressive F1-Optimized Weights')
plt.xticks(range(n_classes))
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Class distribution: {class_counts}')
print('\nAggressive F1-optimized weights:')
for name, weights in f1_strategies.items():
    print(f'{name:15}: {weights}')

# Используем самую агрессивную стратегию
class_weights = f1_strategies['f1_aggressive']
print(f'\nUsing aggressive F1 weights: {class_weights}')

## 2. Реализация блоков

Реализуйте три типа блоков:
- **Bottleneck**: dim → dim//4 → dim (сужение)
- **Inverted Bottleneck**: dim → dim*4 → dim (расширение)
- **Regular**: dim → hidden_dim → dim (обычный)

In [ ]:
from abc import ABC, abstractmethod

class BaseMLPBlock(nn.Module, ABC):
    """Базовый класс для MLP блока"""
    def __init__(self, dim, activation='gelu', dropout=0.0):
        super().__init__()
        self.dim = dim
        self.activation = {'relu': nn.ReLU(), 'gelu': nn.GELU(), 'swish': nn.SiLU()}.get(activation, nn.GELU())
        self.dropout = nn.Dropout(dropout) if dropout > 0 else None
    
    @abstractmethod
    def forward(self, x):
        pass

class BottleneckBlock(BaseMLPBlock):
    """
    Bottleneck блок: dim → dim//4 → dim
    
    Сужает размерность в 4 раза, затем восстанавливает.
    Использует residual connection для стабильного обучения.
    """
    def __init__(self, dim, activation='gelu', dropout=0.0):
        super().__init__(dim, activation, dropout)
        
        # Bottleneck dimension (сужение в 4 раза)
        self.bottleneck_dim = max(dim // 4, 1)
        
        # Линейные слои: dim → bottleneck_dim → dim
        self.fc1 = nn.Linear(self.dim, self.bottleneck_dim)
        self.fc2 = nn.Linear(self.bottleneck_dim, self.dim)
    
    def forward(self, x):
        identity = x
        
        # Bottleneck pathway
        out = self.fc1(x)
        out = self.activation(out)
        if self.dropout:
            out = self.dropout(out)
        out = self.fc2(out)
        
        # Residual connection
        return out + identity

class InvertedBottleneckBlock(BaseMLPBlock):
    """
    Inverted Bottleneck блок: dim → dim*4 → dim
    
    Расширяет размерность в 4 раза, затем сжимает обратно.
    Использует residual connection для стабильного обучения.
    """
    def __init__(self, dim, expansion_factor=4, activation='gelu', dropout=0.0):
        super().__init__(dim, activation, dropout)
        
        # Expanded dimension (расширение в 4 раза)
        self.expanded_dim = dim * expansion_factor
        
        # Линейные слои: dim → expanded_dim → dim
        self.fc1 = nn.Linear(self.dim, self.expanded_dim)
        self.fc2 = nn.Linear(self.expanded_dim, self.dim)
    
    def forward(self, x):
        identity = x
        
        # Inverted bottleneck pathway
        out = self.fc1(x)
        out = self.activation(out)
        if self.dropout:
            out = self.dropout(out)
        out = self.fc2(out)
        
        # Residual connection
        return out + identity

class RegularBlock(BaseMLPBlock):
    """
    Regular блок: dim → hidden_dim → dim
    
    Обычный двухслойный MLP с residual connection.
    hidden_dim по умолчанию равен dim * 2.
    """
    def __init__(self, dim, hidden_dim=None, activation='gelu', dropout=0.0):
        super().__init__(dim, activation, dropout)
        
        # Hidden dimension (по умолчанию в 2 раза больше)
        self.hidden_dim = hidden_dim if hidden_dim else dim * 2
        
        # Линейные слои: dim → hidden_dim → dim
        self.fc1 = nn.Linear(self.dim, self.hidden_dim)
        self.fc2 = nn.Linear(self.hidden_dim, self.dim)
    
    def forward(self, x):
        identity = x
        
        # Regular pathway
        out = self.fc1(x)
        out = self.activation(out)
        if self.dropout:
            out = self.dropout(out)
        out = self.fc2(out)
        
        # Residual connection
        return out + identity

# Тестируем блоки
print('✓ Блоки успешно определены!')
print()

# Проверим размерности
test_x = torch.randn(4, 64)
print('Тестирование блоков с размерностью 64:')
print(f'Input shape: {test_x.shape}')

bottleneck = BottleneckBlock(64)
print(f'BottleneckBlock output: {bottleneck(test_x).shape}')

inverted = InvertedBottleneckBlock(64)
print(f'InvertedBottleneckBlock output: {inverted(test_x).shape}')

regular = RegularBlock(64)
print(f'  RegularBlock output: {regular(test_x).shape}')

# Подсчитаем параметры
print()
print('Количество параметров:')
print(f'BottleneckBlock: {sum(p.numel() for p in bottleneck.parameters()):,}')
print(f'InvertedBottleneckBlock: {sum(p.numel() for p in inverted.parameters()):,}')
print(f'RegularBlock: {sum(p.numel() for p in regular.parameters()):,}')

## 3. Multi-Branch модель

Реализуйте модель с тремя параллельными ветками.

**Архитектура**:
```
         Input
           |
      projection
           |
      ┌────┼────┐
      │    │    │
  Bottleneck  Inverted  Regular
   Branch      Branch    Branch
      │    │    │
      └────┼────┘
           |
      Concatenate/Sum
           |
      projection
           |
        Output
```

In [ ]:
class MultiBranchMLP(nn.Module):
    """
    Multi-Branch MLP с тремя параллельными ветками и identity skip connection.
    
    Args:
        input_dim: размерность входа
        hidden_dim: размерность скрытых слоев
        output_dim: размерность выхода (число классов)
        num_blocks: количество блоков в каждой ветке
        dropout: вероятность dropout
        combine_mode: способ объединения веток ('concat' или 'sum')
        use_skip_connection: использовать ли identity skip connection
    """
    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        num_blocks=4,
        dropout=0.1,
        combine_mode='concat',
        use_skip_connection=True
    ):
        super().__init__()
        self.output_dim = output_dim
        self.combine_mode = combine_mode
        self.use_skip_connection = use_skip_connection
        
        # Входная проекция
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.input_activation = nn.GELU()
        self.input_dropout = nn.Dropout(dropout)
        
        # Создаем три ветки (branches)
        # Branch 1: num_blocks блоков BottleneckBlock
        self.bottleneck_branch = nn.ModuleList([
            BottleneckBlock(hidden_dim, dropout=dropout) 
            for _ in range(num_blocks)
        ])
        
        # Branch 2: num_blocks блоков InvertedBottleneckBlock
        self.inverted_branch = nn.ModuleList([
            InvertedBottleneckBlock(hidden_dim, dropout=dropout) 
            for _ in range(num_blocks)
        ])
        
        # Branch 3: num_blocks блоков RegularBlock
        self.regular_branch = nn.ModuleList([
            RegularBlock(hidden_dim, dropout=dropout) 
            for _ in range(num_blocks)
        ])
        
        # Выходная проекция
        if combine_mode == 'concat':
            combined_dim = hidden_dim * 3
        else:  # 'sum'
            combined_dim = hidden_dim
            
        self.output_proj = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )
        
        # Skip connection projection (если размерности не совпадают)
        if use_skip_connection and input_dim != output_dim:
            self.skip_proj = nn.Linear(input_dim, output_dim)
        else:
            self.skip_proj = None
    
    def forward(self, x):
        # Сохраняем вход для skip connection
        identity = x
        
        # Входная проекция
        x = self.input_proj(x)
        x = self.input_activation(x)
        x = self.input_dropout(x)
        
        # Пропускаем через каждую ветку
        x_bottleneck = x
        for block in self.bottleneck_branch:
            x_bottleneck = block(x_bottleneck)
            
        x_inverted = x
        for block in self.inverted_branch:
            x_inverted = block(x_inverted)
            
        x_regular = x
        for block in self.regular_branch:
            x_regular = block(x_regular)
        
        # Объединяем результаты (concat или sum)
        if self.combine_mode == 'concat':
            combined = torch.cat([x_bottleneck, x_inverted, x_regular], dim=-1)
        else:  # 'sum'
            combined = x_bottleneck + x_inverted + x_regular
        
        # Выходная проекция
        output = self.output_proj(combined)
        
        # Добавляем identity skip connection
        if self.use_skip_connection:
            if self.skip_proj is not None:
                identity = self.skip_proj(identity)
            output = output + identity
        
        return output

print('Multi-Branch модель с identity skip connection определена!')

## 4. Код обучение

In [ ]:
def train_model(
    model,
    dm,
    class_weights=None,
    max_epochs=50,
    lr=1e-3,
    optimizer_type='adam',
    optimizer_kwargs=None 
):
    """
    Обучает модель с weighted loss.
    
    Args:
        model: модель для обучения
        dm: DataModule
        class_weights: веса классов для weighted loss (numpy array или None)
        max_epochs: количество эпох
        lr: learning rate
        optimizer_type: тип оптимизатора ('adam', 'adamw', 'sgd')
        optimizer_kwargs: дополнительные параметры оптимизатора
    
    Returns:
        dict с метриками
    """
    # Создаем loss function
    if class_weights is not None:
        loss_fn = nn.CrossEntropyLoss(weight=torch.FloatTensor(class_weights))
    else:
        loss_fn = nn.CrossEntropyLoss()
    
    lightning_model = BaseLightningModule(
        model=model,
        loss_fn=loss_fn,
        optimizer_type=optimizer_type,
        learning_rate=lr,
        optimizer_kwargs=optimizer_kwargs or {},  # Передаем параметры
        task_type='multiclass'
    )
    
    trainer = pl.Trainer(
        max_epochs=max_epochs,
        enable_checkpointing=False,
        logger=False,
        enable_progress_bar=True,
        enable_model_summary=False,
        callbacks=[
            pl.callbacks.EarlyStopping(
                monitor='val_f1_macro',
                patience=15,
                mode='max',
                min_delta=0.001
            )
        ]
    )
    
    trainer.fit(lightning_model, dm)
    
    metrics = trainer.callback_metrics
    return {
        'val_acc': metrics.get('val_accuracy', 0).item(),
        'val_f1': metrics.get('val_f1_macro', 0).item(),
        'val_loss': metrics.get('val_loss', 0).item()
    }

## 5. Подбор гиперпараметров

In [ ]:
# Подбор гиперпараметров
best_f1 = 0
best_params = {}

# Параметры для поиска
hidden_dims = [64, 128, 256]
num_blocks_list = [2, 4, 6]
lrs = [1e-2, 1e-3, 1e-4]
optimizers = ['adam', 'adamw', 'sgd']

print("Начинаем подбор гиперпараметров...")

for hidden_dim in hidden_dims:
    for num_blocks in num_blocks_list:
        for lr in lrs:
            for optimizer_type in optimizers:
                
                print(f"\nТестируем: hidden_dim={hidden_dim}, blocks={num_blocks}, lr={lr}, optimizer={optimizer_type}")
                
                model = MultiBranchMLP(
                    input_dim=dm.input_dim,
                    hidden_dim=hidden_dim,
                    output_dim=dm.n_classes,
                    num_blocks=num_blocks,
                    dropout=0.1,
                    combine_mode='concat'
                )
                
                # Настройка параметров оптимизатора для SGD с momentum
                optimizer_kwargs = {}
                if optimizer_type == 'sgd':
                    optimizer_kwargs = {
                        'momentum': 0.9,
                        'nesterov': True
                    }
                    print(f"  SGD с momentum={optimizer_kwargs['momentum']}, nesterov={optimizer_kwargs['nesterov']}")
                
                try:
                    results = train_model(
                        model,
                        dm,
                        class_weights=class_weights,
                        max_epochs=100, 
                        lr=lr,
                        optimizer_type=optimizer_type,
                        optimizer_kwargs=optimizer_kwargs
                    )
                    
                    print(f"Результаты: F1={results['val_f1']:.4f}, Acc={results['val_acc']:.4f}")
                    
                    if results['val_f1'] > best_f1:
                        best_f1 = results['val_f1']
                        best_params = {
                            'hidden_dim': hidden_dim,
                            'num_blocks': num_blocks,
                            'lr': lr,
                            'optimizer': optimizer_type,
                            'optimizer_kwargs': optimizer_kwargs
                        }
                        print(f"*** Новый лучший результат! ***")
                        
                except Exception as e:
                    print(f"Ошибка: {e}")
                    continue

print(f"\nЛучшие параметры: {best_params}")
print(f"Лучший F1: {best_f1:.4f}")

## 6. Итоговая модель

In [ ]:
# Обучаем итоговую модель с лучшими гиперпараметрами

final_model = MultiBranchMLP(
    input_dim=dm.input_dim,
    hidden_dim=best_params.get('hidden_dim', 256),
    output_dim=dm.n_classes,
    num_blocks=best_params.get('num_blocks', 6),
    dropout=0.1,
    combine_mode='concat'
)

final_results = train_model(
    final_model,
    dm,
    class_weights=class_weights,
    max_epochs=100,
    lr=best_params.get('lr', 1e-3),
    optimizer_type=best_params.get('optimizer', 'adamw'),
)

print(f'\n=== Итоговые результаты ===')
print(f"F1 score: {final_results['val_f1']:.4f}")
print(f"Accuracy: {final_results['val_acc']:.4f}")